<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Introduction to Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">1. Apache Spark Runtime Architecture</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 1.2 Lecture - Spark Runtime Architecture
### Duration - 13 Minutes

>  *How Spark plans, distributes, and monitors work across a cluster*

This lecture examines the components of a running Spark application - the Driver, Cluster Manager, Workers, and Executors - and how Spark breaks work into jobs, stages, and tasks that you can observe in the Spark UI.

## Learning Objectives

**By the end of this lecture, you will be able to:**

- Identify the roles of the Driver, Workers, and Executors in Spark applications
- Explain how the `SparkSession` serves as the unified entry point for Spark applications
- Describe how a Spark application is decomposed into jobs, stages, and tasks
- Use the Spark UI to monitor application progress, DAG visualizations, and resource allocation

## A. The Runtime Architecture at a Glance

<p style="font-size: 1em; line-height: 1.6; color: #333">A Spark application is a coordinated set of processes. The Driver plans the work, the Cluster Manager allocates resources, and Executors on Worker nodes do the computation. Click the bar below to reveal the diagram.</p>

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Spark runtime architecture diagram</strong>
      </div>
    </div>
  </summary>
  <div style="width:100%; max-width:960px; margin:8px 0">
  <svg viewBox="0 0 960 440" style="width:100%; height:auto" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif">
    <defs>
      <marker id="rt-arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L8,3 L0,6 Z" fill="#618794"/></marker>
      <marker id="rt-arrow-start" markerWidth="10" markerHeight="10" refX="0" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M8,0 L0,3 L8,6 Z" fill="#618794"/></marker>
    </defs>
    <line x1="140" y1="222" x2="176" y2="222" stroke="#618794" stroke-width="2" marker-start="url(#rt-arrow-start)" marker-end="url(#rt-arrow)"/>
    <line x1="376" y1="225" x2="426" y2="225" stroke="#618794" stroke-width="2" marker-start="url(#rt-arrow-start)" marker-end="url(#rt-arrow)"/>
    <line x1="600" y1="215" x2="686" y2="122" stroke="#618794" stroke-width="2" marker-end="url(#rt-arrow)"/>
    <line x1="600" y1="245" x2="686" y2="350" stroke="#618794" stroke-width="2" marker-end="url(#rt-arrow)"/>
    <path d="M295,150 C 420,45 600,45 705,108" fill="none" stroke="#618794" stroke-width="2" stroke-dasharray="6 4"/>
    <path d="M295,300 C 420,415 600,415 705,352" fill="none" stroke="#618794" stroke-width="2" stroke-dasharray="6 4"/>
    <rect x="16" y="190" width="124" height="64" rx="32" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="78" y="226" text-anchor="middle" font-size="15" fill="#333">Client / Job</text>
    <rect x="180" y="150" width="194" height="150" rx="8" fill="#DCE0E2" stroke="#618794" stroke-width="2"/>
    <text x="192" y="172" font-size="14" font-weight="bold" fill="#1B5162">Driver</text>
    <rect x="200" y="188" width="154" height="96" rx="6" fill="#FFFFFF" stroke="#618794" stroke-width="1.5"/>
    <text x="277" y="230" text-anchor="middle" font-size="14" fill="#333">SparkSession /</text>
    <text x="277" y="250" text-anchor="middle" font-size="14" fill="#333">SparkContext</text>
    <rect x="430" y="192" width="170" height="72" rx="8" fill="#FFCC66" stroke="#CD7F32" stroke-width="2"/>
    <text x="515" y="222" text-anchor="middle" font-size="14" fill="#333">Cluster Manager</text>
    <text x="515" y="244" text-anchor="middle" font-size="14" fill="#333">(Master)</text>
    <rect x="690" y="40" width="252" height="160" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="704" y="62" font-size="14" font-weight="bold" fill="#1B5162">Worker</text>
    <rect x="708" y="78" width="216" height="104" rx="6" fill="#F9F7F4" stroke="#618794" stroke-width="1.5"/>
    <text x="718" y="98" font-size="13" font-weight="bold" fill="#1B5162">Executor + Cache</text>
    <rect x="724" y="110" width="90" height="56" rx="6" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="769" y="142" text-anchor="middle" font-size="14" fill="#333">Task</text>
    <rect x="820" y="110" width="90" height="56" rx="6" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="865" y="142" text-anchor="middle" font-size="14" fill="#333">Task</text>
    <rect x="690" y="270" width="252" height="160" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="704" y="292" font-size="14" font-weight="bold" fill="#1B5162">Worker</text>
    <rect x="708" y="308" width="216" height="104" rx="6" fill="#F9F7F4" stroke="#618794" stroke-width="1.5"/>
    <text x="718" y="328" font-size="13" font-weight="bold" fill="#1B5162">Executor + Cache</text>
    <rect x="724" y="340" width="90" height="56" rx="6" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="769" y="372" text-anchor="middle" font-size="14" fill="#333">Task</text>
    <rect x="820" y="340" width="90" height="56" rx="6" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="865" y="372" text-anchor="middle" font-size="14" fill="#333">Task</text>
    <text x="480" y="424" text-anchor="middle" font-size="12" fill="#666" font-style="italic">Dashed lines: the Driver schedules tasks on Executors directly</text>
  </svg>
  </div>
</details>

<p style="font-size: 1em; line-height: 1.6; color: #333">The four roles:</p>

- **Driver** - the brain of a Spark application, responsible for planning and coordinating execution
- **Cluster Manager (Master)** - manages cluster resources and allocates them to the Driver
- **Workers** - nodes in the cluster that host Executors
- **Executors** - processes on Worker nodes that execute tasks assigned by the Driver
<br/>
<details>
<summary style="cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for More Details</strong>
</div>
</div>
</summary>
<div style="border-left: 4px solid #1B5162; padding: 4px 24px 14px 24px; margin: -16px 0 16px 0">
<div style="font-size: 1.2em; font-weight: bold; color: #1B5162; margin: 12px 0 4px 0">How the Pieces Interact</div>
<ul style="font-size: 1.1em; line-height: 1.45; color: #333; margin: 0 0 10px 0; padding-left: 24px">
<li style="margin: 4px 0">The client (a notebook, a job, or an application) submits work to the Driver. The Driver negotiates with the Cluster Manager for Executor resources, then communicates directly with those Executors to schedule tasks and collect results - shown as the dashed lines in the diagram.</li>
<li style="margin: 4px 0">Each Executor holds a cache, a region of memory for partitions you explicitly persist. Cached partitions can be reused across jobs without recomputation.</li>
<li style="margin: 4px 0">In Databricks, the Cluster Manager role is handled by the platform itself: when you create a cluster, Databricks provisions a Driver node and Worker nodes and wires them together for you.</li>
</ul>
<div style="font-size: 1.2em; font-weight: bold; color: #1B5162; margin: 14px 0 4px 0">Spark Connect: a Thinner Client</div>
<ul style="font-size: 1.1em; line-height: 1.45; color: #333; margin: 0 0 10px 0; padding-left: 24px">
<li style="margin: 4px 0">Spark 4 promotes <strong>Spark Connect</strong>, a client-server protocol in which your application holds only a lightweight client; the Driver runs remotely and receives query plans over gRPC.</li>
<li style="margin: 4px 0">This is how Databricks Serverless compute and Databricks Connect work: your notebook or IDE is the thin client, and the heavy lifting happens on the server side. The DataFrame code you write is identical either way.</li>
</ul>
</div>
</details>

## B. The Spark Driver

<p style="font-size: 1em; line-height: 1.6; color: #333">The Driver plans, coordinates, and manages task execution. It:</p>

- Creates the `SparkSession`, the entry point for all Spark applications
- Analyzes the Spark application and constructs a Directed Acyclic Graph (DAG)
- Schedules and distributes tasks to Executors for execution
- Monitors the progress of tasks and handles failures
- Returns results to the client

## C. The SparkSession

<p style="font-size: 1em; line-height: 1.6; color: #333">Introduced in Spark 2.0 as the unified entry point for all Spark functionality, replacing the separate <code>SparkContext</code>, <code>SQLContext</code>, <code>HiveContext</code>, and <code>StreamingContext</code> objects that earlier versions required.</p>

<div class="code-block" data-language="python">
from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .appName("MySparkApplication")
    .config("spark.some.config.option", "some-value")
    .getOrCreate())
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism-tomorrow.min.css" rel="stylesheet" />
<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" id="prism-light" disabled />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'python';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);

        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<button class="theme-toggle" style="position:absolute;top:8px;left:8px;padding:4px 10px;font-size:12px;background:#333;color:#fff;border:1px solid #666;border-radius:4px;cursor:pointer;z-index:10;">◑</button>' +
                '<pre class="code-pre" data-theme="dark" style="background:#1e1e1e;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';

        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);

        var toggleBtn = block.querySelector('.theme-toggle');
        toggleBtn.onclick = function() {
            var pre = block.querySelector('.code-pre');
            var isDark = pre.getAttribute('data-theme') !== 'light';
            pre.setAttribute('data-theme', isDark ? 'light' : 'dark');
            pre.style.background = isDark ? '#f8f8f8' : '#1e1e1e';
            this.textContent = isDark ? '◐' : '◑';
            this.style.background = isDark ? '#ddd' : '#333';
            this.style.color = isDark ? '#333' : '#fff';
            document.getElementById('prism-light').disabled = !isDark;
            document.querySelector('link[href*="prism-tomorrow"]').disabled = isDark;
            Prism.highlightElement(codeEl);
        };
    });
})();
</script>

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128161;</span>
<div>
<strong style="color: #00695c; font-size: 1.1em">Already Created in Databricks</strong>
<p style="margin: 8px 0 0 0; color: #333">In Databricks notebooks the <code>SparkSession</code> is created for you automatically and made available as the variable <code>spark</code>. You will almost never construct one yourself.</p>
</div>
</div>
</div>

## D. Executors

<p style="font-size: 1em; line-height: 1.6; color: #333">Executors are the processes that do the actual work:</p>

- Run on Worker nodes in a Spark cluster and host **Tasks** (for I/O and data processing)
- Typically each Worker node runs one Executor, which has its own:
  - RAM cache
  - Disk cache
  - CPU cores
- Interact with the Driver for task coordination and data transfer

## E. The Spark DAG

<p style="font-size: 1em; line-height: 1.6; color: #333">Spark represents each job as a <strong>Directed Acyclic Graph</strong> of stages:</p>

- Spark jobs are broken down into **Stages** - groups of tasks that can be executed in parallel
- Computations flow in one direction through the stages (the *Directed* bit)
- Stages never loop back, ensuring the job terminates (the *Acyclic* bit)
- Stages are organized into a dependency graph for execution flow (the *Graph* bit)

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Example DAG with two stages</strong>
      </div>
    </div>
  </summary>
  <div style="width:100%; max-width:560px; margin:8px 0">
  <svg viewBox="0 0 700 540" style="width:100%; height:auto" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif">
    <defs>
      <marker id="dag-arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L8,3 L0,6 Z" fill="#618794"/></marker>
    </defs>
    <line x1="350" y1="104" x2="350" y2="130" stroke="#618794" stroke-width="2" marker-end="url(#dag-arrow)"/>
    <line x1="350" y1="180" x2="350" y2="206" stroke="#618794" stroke-width="2" marker-end="url(#dag-arrow)"/>
    <line x1="350" y1="270" x2="350" y2="328" stroke="#618794" stroke-width="2" marker-end="url(#dag-arrow)"/>
    <line x1="350" y1="414" x2="350" y2="440" stroke="#618794" stroke-width="2" marker-end="url(#dag-arrow)"/>
    <rect x="200" y="20" width="300" height="250" rx="8" fill="#DCE0E2" stroke="#618794" stroke-width="2"/>
    <text x="212" y="44" font-size="15" font-weight="bold" fill="#1B5162">Stage 1</text>
    <rect x="250" y="56" width="200" height="48" rx="6" fill="#FFFFFF" stroke="#618794" stroke-width="1.5"/>
    <text x="350" y="86" text-anchor="middle" font-size="14" fill="#333">Scan source files</text>
    <rect x="250" y="132" width="200" height="48" rx="6" fill="#FFFFFF" stroke="#618794" stroke-width="1.5"/>
    <text x="350" y="162" text-anchor="middle" font-size="14" fill="#333">Filter rows</text>
    <rect x="250" y="208" width="200" height="48" rx="6" fill="#FFFFFF" stroke="#618794" stroke-width="1.5"/>
    <text x="350" y="238" text-anchor="middle" font-size="14" fill="#333">Project columns</text>
    <rect x="200" y="328" width="300" height="182" rx="8" fill="#FFCC66" stroke="#CD7F32" stroke-width="2"/>
    <text x="212" y="352" font-size="15" font-weight="bold" fill="#8a5a12">Stage 2</text>
    <rect x="250" y="364" width="200" height="48" rx="6" fill="#FFFFFF" stroke="#CD7F32" stroke-width="1.5"/>
    <text x="350" y="394" text-anchor="middle" font-size="14" fill="#333">Aggregate by key</text>
    <rect x="250" y="440" width="200" height="48" rx="6" fill="#FFFFFF" stroke="#CD7F32" stroke-width="1.5"/>
    <text x="350" y="470" text-anchor="middle" font-size="14" fill="#333">Return results</text>
    <rect x="282" y="288" width="136" height="24" rx="4" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1"/>
    <text x="350" y="305" text-anchor="middle" font-size="12" fill="#333">shuffle (exchange)</text>
  </svg>
  </div>
</details><br/>
<details>
<summary style="cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for More Details</strong>
</div>
</div>
</summary>
<div style="border-left: 4px solid #1B5162; padding: 4px 24px 14px 24px; margin: -16px 0 16px 0">
<div style="font-size: 1.2em; font-weight: bold; color: #1B5162; margin: 12px 0 4px 0">Why Stages Exist</div>
<ul style="font-size: 1.1em; line-height: 1.45; color: #333; margin: 0 0 10px 0; padding-left: 24px">
<li style="margin: 4px 0">Operations that can be applied to each partition independently (filter, project) are pipelined together into a single stage and execute in parallel across partitions.</li>
<li style="margin: 4px 0">A stage boundary appears wherever data must be redistributed across the cluster - the <em>shuffle</em>. Aggregating by key requires all rows for a key to be on the same Executor, so a new stage begins after the exchange.</li>
<li style="margin: 4px 0">In the Spark UI you will see this DAG rendered for every job, with each stage's operators and the dependencies between stages. Section 3 examines the shuffle in depth.</li>
</ul>
</div>
</details>

## F. Spark Application Execution

<p style="font-size: 1em; line-height: 1.6; color: #333">A Spark application decomposes into a hierarchy: an application runs one or more <strong>jobs</strong>, each job consists of one or more <strong>stages</strong>, and each stage executes many <strong>tasks</strong> in parallel.</p>

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Application, job, stage, and task hierarchy</strong>
      </div>
    </div>
  </summary>
  <div style="width:100%; max-width:820px; margin:8px 0">
  <svg viewBox="0 0 786 310" style="width:100%; height:auto" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif">
    <defs>
      <marker id="hier-arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L8,3 L0,6 Z" fill="#618794"/></marker>
    </defs>
    <line x1="166" y1="202" x2="222" y2="117" stroke="#618794" stroke-width="2" marker-end="url(#hier-arrow)"/>
    <line x1="166" y1="202" x2="222" y2="275" stroke="#618794" stroke-width="2" marker-end="url(#hier-arrow)"/>
    <line x1="346" y1="117" x2="432" y2="85" stroke="#618794" stroke-width="2" marker-end="url(#hier-arrow)"/>
    <line x1="346" y1="117" x2="432" y2="175" stroke="#618794" stroke-width="2" marker-end="url(#hier-arrow)"/>
    <line x1="556" y1="85" x2="642" y2="46" stroke="#618794" stroke-width="2" marker-end="url(#hier-arrow)"/>
    <line x1="556" y1="85" x2="642" y2="104" stroke="#618794" stroke-width="2" marker-end="url(#hier-arrow)"/>
    <line x1="556" y1="85" x2="642" y2="162" stroke="#618794" stroke-width="2" marker-end="url(#hier-arrow)"/>
    <rect x="16" y="170" width="150" height="64" rx="8" fill="#DCE0E2" stroke="#618794" stroke-width="2"/>
    <text x="91" y="197" text-anchor="middle" font-size="14" font-weight="bold" fill="#1B5162">Spark</text>
    <text x="91" y="217" text-anchor="middle" font-size="14" font-weight="bold" fill="#1B5162">Application</text>
    <rect x="226" y="92" width="120" height="50" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="286" y="123" text-anchor="middle" font-size="14" fill="#333">Job 1</text>
    <rect x="226" y="250" width="120" height="50" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="286" y="281" text-anchor="middle" font-size="14" fill="#333">Job 2</text>
    <rect x="436" y="60" width="120" height="50" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="496" y="91" text-anchor="middle" font-size="14" fill="#333">Stage 1</text>
    <rect x="436" y="150" width="120" height="50" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="496" y="181" text-anchor="middle" font-size="14" fill="#333">Stage 2</text>
    <rect x="646" y="24" width="120" height="44" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="706" y="52" text-anchor="middle" font-size="14" fill="#333">Task 1</text>
    <rect x="646" y="82" width="120" height="44" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="706" y="110" text-anchor="middle" font-size="14" fill="#333">Task 2</text>
    <rect x="646" y="140" width="120" height="44" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
    <text x="706" y="168" text-anchor="middle" font-size="14" fill="#333">Task 3</text>
  </svg>
  </div>
</details>

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128221;</span>
<div>
<strong style="color: #4a148c; font-size: 1.1em">Shared Nothing</strong>
<p style="margin: 8px 0 0 0; color: #333">Tasks within a stage run in parallel and do not communicate with each other - a model known as <em>shared nothing</em> execution. One task processes one partition of data.</p>
</div>
</div>
</div>

## G. The Spark UI

<p style="font-size: 1em; line-height: 1.6; color: #333">Spark provides web user interfaces for monitoring and management, including:</p>

- **Application UI** - per application (one per `SparkSession`)
  - Application progress and task execution
  - DAG visualization and stage details
  - Resource usage and performance metrics
- **Master UI** - per cluster
  - Worker node status, health, and cluster-wide resource allocation
  - Shows all running applications and available resources

<p style="font-size: 1em; line-height: 1.6; color: #333">In Databricks, the Spark UI is available from the cluster page (<strong>Compute</strong> -> your cluster -> <strong>Spark UI</strong>) and from the <strong>View</strong> links that appear under each executed notebook cell. The demo that follows explores the <strong>Jobs</strong>, <strong>Executors</strong>, and <strong>Storage</strong> tabs.</p>

## H. Spark Clusters in Databricks

The Databricks platform offers several options for running Spark workloads:

| Compute Type | Description |
|--------------|-------------|
| All-Purpose Clusters | Interactive clusters that support notebooks, jobs, and dashboards with configurable auto-termination |
| Job Clusters | Ephemeral clusters that start when a job runs and terminate automatically upon completion, optimized for non-interactive workloads |
| SQL Warehouses | Optimized compute for SQL query performance with fast startup and auto-scaling to balance cost and performance |
| Serverless | Fully managed compute where Databricks operates the cluster; your notebook connects as a thin client via Spark Connect |

## Summary

- The Driver plans and coordinates; the Cluster Manager allocates resources; Executors on Workers run tasks
- The `SparkSession` (available as `spark` in Databricks) is the unified entry point for all Spark functionality
- Applications decompose into jobs, jobs into stages, and stages into parallel shared-nothing tasks
- Stage boundaries occur at shuffles, where data is redistributed across the cluster
- The Spark UI visualizes jobs, stages, the DAG, executors, and cached data

## Knowledge Check

Answer the questions below to check your understanding of this lecture.

<div id="mcqRoot"></div>

<script>
(function() {
    var questions = [
        {
                "question": "A ____ node in a Spark cluster hosts Executors.",
                "options": [
                        {
                                "id": "a",
                                "text": "Driver"
                        },
                        {
                                "id": "b",
                                "text": "Worker"
                        },
                        {
                                "id": "c",
                                "text": "Cluster Manager"
                        },
                        {
                                "id": "d",
                                "text": "Master"
                        }
                ],
                "correct": "b",
                "explanation": "Workers are the nodes in the cluster that host Executor processes. The Driver plans work, and the Cluster Manager (Master) allocates resources."
        },
        {
                "question": "A Spark job (DAG) consists of ____, which consist of ____ that can be executed in parallel.",
                "options": [
                        {
                                "id": "a",
                                "text": "Executors, Stages"
                        },
                        {
                                "id": "b",
                                "text": "Executors, Tasks"
                        },
                        {
                                "id": "c",
                                "text": "Stages, Tasks"
                        },
                        {
                                "id": "d",
                                "text": "Tasks, Stages"
                        }
                ],
                "correct": "c",
                "explanation": "Jobs are broken into stages at shuffle boundaries, and each stage executes many tasks in parallel - one task per partition."
        },
        {
                "question": "Which component in the Spark runtime architecture is responsible for planning the execution of a Spark job and monitoring its progress?",
                "options": [
                        {
                                "id": "a",
                                "text": "Executor"
                        },
                        {
                                "id": "b",
                                "text": "Worker"
                        },
                        {
                                "id": "c",
                                "text": "Cluster Manager"
                        },
                        {
                                "id": "d",
                                "text": "Driver"
                        }
                ],
                "correct": "d",
                "explanation": "The Driver analyzes the application, constructs the DAG, schedules tasks to Executors, monitors progress, and returns results to the client."
        }
];
    
    var currentIndex = 0;
    var score = 0;
    var selectedAnswer = null;
    var showingFeedback = false;
    
    function init() {
        currentIndex = 0;
        score = 0;
        selectedAnswer = null;
        showingFeedback = false;
        render();
    }
    
    function render() {
        var root = document.getElementById('mcqRoot');
        var html = '';
        var isComplete = currentIndex >= questions.length;
        
        html += '<div style="font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; width: 100%; margin: 10px 0; padding: 24px; background: #f5f7fa; border-radius: 12px; border: 1px solid #e0e0e0; box-sizing: border-box;">';
        
        // Header
        html += '<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">';
        html += '<div>';
        html += '<div style="font-size: 1.4em; font-weight: 600; color: #333;">📝 Multiple Choice Quiz</div>';
        html += '<div style="font-size: 0.95em; color: #666; margin-top: 4px;">Knowledge Check</div>';
        html += '</div>';
        html += '<div style="display: flex; gap: 12px; align-items: center;">';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e8f5e9; color: #2e7d32;">Score: ' + score + '/' + questions.length + '</span>';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e3f2fd; color: #1976d2;">Q' + (currentIndex + 1) + '/' + questions.length + '</span>';
        html += '</div></div>';
        
        if (isComplete) {
            // Results
            var percentage = Math.round(score / questions.length * 100);
            html += '<div style="text-align: center; padding: 40px 20px; background: #fff; border-radius: 12px; border: 2px solid #e0e0e0;">';
            html += '<div style="font-size: 3em; margin-bottom: 16px;">' + (percentage >= 75 ? '🏆' : percentage >= 50 ? '👍' : '📚') + '</div>';
            html += '<div style="font-size: 1.8em; font-weight: 700; color: #333; margin-bottom: 8px;">Quiz Complete!</div>';
            html += '<div style="font-size: 1.2em; color: #666; margin-bottom: 24px;">You scored ' + score + ' out of ' + questions.length + ' (' + percentage + '%)</div>';
            html += '<button id="retakeBtn" style="padding: 14px 32px; background: #1976d2; color: white; border: none; border-radius: 8px; cursor: pointer; font-size: 1.1em; font-weight: 600;">Retake Quiz</button>';
            html += '</div>';
        } else {
            var q = questions[currentIndex];
            
            // Question
            html += '<div style="background: #fff; border-radius: 12px; border: 2px solid #e0e0e0; padding: 24px; margin-bottom: 20px;">';
            html += '<div style="font-size: 1.15em; color: #333; line-height: 1.6;">' + q.question + '</div>';
            html += '</div>';
            
            // Options
            html += '<div style="display: flex; flex-direction: column; gap: 12px; margin-bottom: 20px;">';
            
            q.options.forEach(function(opt) {
                var isSelected = selectedAnswer === opt.id;
                var isCorrect = opt.id === q.correct;
                var bgColor = '#fff';
                var borderColor = '#e0e0e0';
                var textColor = '#333';
                
                if (showingFeedback) {
                    if (isCorrect) {
                        bgColor = '#e8f5e9';
                        borderColor = '#4caf50';
                        textColor = '#2e7d32';
                    } else if (isSelected && !isCorrect) {
                        bgColor = '#ffebee';
                        borderColor = '#f44336';
                        textColor = '#c62828';
                    }
                } else if (isSelected) {
                    bgColor = '#e3f2fd';
                    borderColor = '#1976d2';
                    textColor = '#1565c0';
                }
                
                var cursor = showingFeedback ? 'default' : 'pointer';
                
                html += '<div class="mcq-option" data-id="' + opt.id + '" style="display: flex; align-items: center; gap: 16px; padding: 16px 20px; background: ' + bgColor + '; border: 2px solid ' + borderColor + '; border-radius: 8px; cursor: ' + cursor + '; transition: all 0.15s ease;">';
                html += '<div style="width: 32px; height: 32px; border: 2px solid ' + borderColor + '; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: 700; color: ' + textColor + '; background: ' + (isSelected || (showingFeedback && isCorrect) ? borderColor : 'transparent') + '; color: ' + (isSelected || (showingFeedback && isCorrect) ? '#fff' : textColor) + ';">' + opt.id.toUpperCase() + '</div>';
                html += '<div style="flex: 1; font-size: 1em; color: ' + textColor + ';">' + opt.text + '</div>';
                if (showingFeedback && isCorrect) {
                    html += '<span style="color: #4caf50; font-size: 1.2em;">✓</span>';
                } else if (showingFeedback && isSelected && !isCorrect) {
                    html += '<span style="color: #f44336; font-size: 1.2em;">✗</span>';
                }
                html += '</div>';
            });
            
            html += '</div>';
            
            // Feedback
            if (showingFeedback) {
                var isCorrect = selectedAnswer === q.correct;
                html += '<div style="background: ' + (isCorrect ? '#e8f5e9' : '#fff3e0') + '; border: 2px solid ' + (isCorrect ? '#4caf50' : '#ff9800') + '; border-radius: 8px; padding: 16px 20px; margin-bottom: 20px;">';
                html += '<div style="font-weight: 600; color: ' + (isCorrect ? '#2e7d32' : '#e65100') + '; margin-bottom: 8px;">' + (isCorrect ? '✓ Correct!' : '✗ Not quite') + '</div>';
                html += '<div style="color: #333; font-size: 0.95em; line-height: 1.5;">' + q.explanation + '</div>';
                html += '</div>';
            }
            
            // Buttons
            html += '<div style="display: flex; gap: 12px; justify-content: flex-end;">';
            if (!showingFeedback) {
                html += '<button id="submitBtn" style="padding: 12px 28px; background: ' + (selectedAnswer ? '#4caf50' : '#ccc') + '; color: white; border: none; border-radius: 6px; cursor: ' + (selectedAnswer ? 'pointer' : 'not-allowed') + '; font-size: 1em; font-weight: 600;"' + (selectedAnswer ? '' : ' disabled') + '>Submit Answer</button>';
            } else {
                html += '<button id="nextQBtn" style="padding: 12px 28px; background: #1976d2; color: white; border: none; border-radius: 6px; cursor: pointer; font-size: 1em; font-weight: 600;">' + (currentIndex < questions.length - 1 ? 'Next Question' : 'See Results') + '</button>';
            }
            html += '</div>';
        }
        
        html += '</div>';
        
        root.innerHTML = html;
        attachEvents();
    }
    
    function attachEvents() {
        var options = document.querySelectorAll('.mcq-option');
        var submitBtn = document.getElementById('submitBtn');
        var nextQBtn = document.getElementById('nextQBtn');
        var retakeBtn = document.getElementById('retakeBtn');
        
        options.forEach(function(opt) {
            opt.addEventListener('click', function() {
                if (!showingFeedback) {
                    selectedAnswer = this.dataset.id;
                    render();
                }
            });
            
            if (!showingFeedback) {
                opt.addEventListener('mouseenter', function() {
                    if (selectedAnswer !== this.dataset.id) {
                        this.style.borderColor = '#1976d2';
                        this.style.background = '#f5f9ff';
                    }
                });
                opt.addEventListener('mouseleave', function() {
                    if (selectedAnswer !== this.dataset.id) {
                        this.style.borderColor = '#e0e0e0';
                        this.style.background = '#fff';
                    }
                });
            }
        });
        
        if (submitBtn) {
            submitBtn.addEventListener('click', function() {
                if (selectedAnswer) {
                    var q = questions[currentIndex];
                    if (selectedAnswer === q.correct) {
                        score++;
                    }
                    showingFeedback = true;
                    render();
                }
            });
        }
        
        if (nextQBtn) {
            nextQBtn.addEventListener('click', function() {
                currentIndex++;
                selectedAnswer = null;
                showingFeedback = false;
                render();
            });
        }
        
        if (retakeBtn) {
            retakeBtn.addEventListener('click', function() {
                init();
            });
        }
    }
    
    if (typeof NodeList.prototype.forEach !== 'function') {
        NodeList.prototype.forEach = Array.prototype.forEach;
    }
    
    init();
})();
</script>

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>